# Day 26 — Practice SOLUTIONS
### The Uniform Distribution · Module 2 · Topic 2.6

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it tests | Count |
|---|---|---|
| **A** | Predict the output | 8 |
| **B** | Spot & fix the bug | 4 |
| **C** | Write the code | 5 |
| **D** | Challenge | 2 |

**How to work:** for Part A, write your answer down *before* running anything.

Three questions to keep asking yourself:

- Is this **discrete** or **continuous**? (pmf or pdf?)
- Did I give scipy an **endpoint** or a **width**?
- Am I asking about a **point** or a **range**?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import random
import warnings
warnings.simplefilter("ignore")

rng = np.random.default_rng(26)
print("ready")

---
# Part A — Predict the output

Write your answer first. Then run the cell.

### A1

```python
print(stats.uniform(0, 10).pdf(3))
print(stats.uniform(0, 10).pdf(7))
print(stats.uniform(0, 10).pdf(12))
```

Three numbers. What are they?

In [ ]:
print(stats.uniform(0, 10).pdf(3))
print(stats.uniform(0, 10).pdf(7))
print(stats.uniform(0, 10).pdf(12))

**`0.1`, `0.1`, `0.0`**

The density is the same everywhere *inside* the range — that is what "uniform" means — and
zero outside it. The height is 1/(b−a) = 1/10.

Note that `0.1` is **not** the probability of getting 3. It is a density. The probability of
exactly 3 is zero.

In [ ]:
u = stats.uniform(0, 10)
print("pdf is flat inside, zero outside:")
for x in [-1, 0, 3, 7, 10, 12]:
    print(f"  pdf({x:>3}) = {u.pdf(x)}")
print()
print("P(X = exactly 3) =", u.cdf(3) - u.cdf(3), " <- the actual probability")

### A2

```python
u = stats.uniform(0, 0.2)
print(u.pdf(0.1))
print(u.cdf(0.2))
```

One of these is bigger than 1. Which, and is that a bug?

In [ ]:
u = stats.uniform(0, 0.2)
print(u.pdf(0.1))
print(u.cdf(0.2))

**`5.0` and `1.0`.** The pdf is 5, and that is not a bug.

A **probability** can never exceed 1. A **density** can be anything. Here the same amount of
probability has to fit into a width of 0.2, so the height is 1/0.2 = 5. Area = 0.2 × 5 = 1.

A density is probability *per unit of x*, like a price per kilo. £5 per unit over 0.2 units is
£1.

The `cdf` is the one that is a probability, and it correctly tops out at exactly 1.

In [ ]:
for w in [2, 1, 0.5, 0.2, 0.05]:
    d = stats.uniform(0, w)
    print(f"{'U(0, ' + format(w, 'g') + ')':<11} height {d.pdf(w/2):>6.4g}   area {d.cdf(w):.1f}")
print()
print("height x width = 1, always")

### A3

```python
u = stats.uniform(2, 8)
print(u.ppf(0), u.ppf(1))
print(u.mean())
```

What range does this distribution actually cover?

In [ ]:
u = stats.uniform(2, 8)
print(u.ppf(0), u.ppf(1))
print(u.mean())

**It covers 2 to 10, and the mean is 6.**

`stats.uniform(loc, scale)` takes a starting point and a **width**. So `uniform(2, 8)` starts
at 2 and is 8 wide — U(2, 10), not U(2, 8).

If you wanted U(2, 8) you need `stats.uniform(2, 6)`.

**The rule: `U(a, b)` is `stats.uniform(a, b - a)`.**

In [ ]:
print("uniform(2, 8) spans", stats.uniform(2, 8).ppf([0, 1]), "mean", stats.uniform(2, 8).mean())
print("uniform(2, 6) spans", stats.uniform(2, 6).ppf([0, 1]), "mean", stats.uniform(2, 6).mean())
print()
print("U(a, b)  ->  stats.uniform(a, b - a)")

### A4

```python
print(stats.randint(1, 7).pmf(6))
print(stats.randint(1, 6).pmf(6))
```

Both look like a six-sided die. Are they?

In [ ]:
print(stats.randint(1, 7).pmf(6))
print(stats.randint(1, 6).pmf(6))

**`0.1667` and `0.0`.**

`randint(low, high)` excludes `high`. So `randint(1, 7)` gives 1–6 — a real die — and
`randint(1, 6)` gives 1–5, a five-sided die on which rolling a 6 is impossible.

Nothing errors. The second one just quietly has a mean of 3.0 instead of 3.5, and every
probability is a fifth instead of a sixth.

In [ ]:
for hi in [6, 7]:
    d = stats.randint(1, hi)
    faces = [k for k in range(1, 8) if d.pmf(k) > 0]
    print(f"randint(1, {hi}): faces {faces}  mean {d.mean()}  each {d.pmf(1):.4f}")

### A5

```python
random.seed(0)
print([random.randint(1, 6) for _ in range(8)])

rng2 = np.random.default_rng(0)
print(rng2.integers(1, 6, 8))
```

Both are asking for dice rolls. Will both produce 6s?

In [ ]:
random.seed(0)
print([random.randint(1, 6) for _ in range(8)])

rng2 = np.random.default_rng(0)
print(rng2.integers(1, 6, 8))

**Only the first one can.**

Python's own `random.randint(1, 6)` **includes** 6. NumPy's `rng.integers(1, 6)` **excludes**
it — you need `rng.integers(1, 7)`.

Two libraries, two conventions, same-looking call. This is one of the most common off-by-one
bugs in simulation code, and it never raises an error — it just shifts every result.

In [ ]:
random.seed(0)
py = [random.randint(1, 6) for _ in range(20_000)]
rng2 = np.random.default_rng(0)
np_wrong = rng2.integers(1, 6, 20_000)
np_right = np.random.default_rng(0).integers(1, 7, 20_000)

for name, d in [("random.randint(1,6)", py), ("np integers(1,6)", np_wrong),
                ("np integers(1,7)", np_right)]:
    d = np.asarray(d)
    print(f"{name:<22} max {d.max()}  mean {d.mean():.3f}")
print()
print("target mean for a fair die: 3.5")

### A6

```python
rng3 = np.random.default_rng(7)
d = rng3.random(100_000)
print(round(d.mean(), 3), round(d.std(), 4))
```

What two numbers, and where does the second come from?

In [ ]:
rng3 = np.random.default_rng(7)
d = rng3.random(100_000)
print(round(d.mean(), 3), round(d.std(), 4))

**About `0.5` and `0.2886`.**

The mean is obvious — the middle of 0 and 1.

The sd is `(b−a)/√12 = 1/√12 = 0.2887`, and the sample gives 0.2886. That 12 falls out of integrating x² across the
rectangle, and you will not need to derive it. But remember the number: it reappears the
moment anyone adds uniforms together, which happens in Part D.

In [ ]:
print(f"simulated mean {d.mean():.5f}   theory {0.5}")
print(f"simulated sd   {d.std():.5f}   theory 1/sqrt(12) = {1/np.sqrt(12):.5f}")
print(f"simulated var  {d.var():.5f}   theory 1/12       = {1/12:.5f}")

### A7

```python
u = stats.uniform(0, 1)
print(u.cdf(0.7) - u.cdf(0.3))
print(u.cdf(0.5) - u.cdf(0.5))
```

Predict both, and say what the second one tells you.

In [ ]:
u = stats.uniform(0, 1)
print(u.cdf(0.7) - u.cdf(0.3))
print(u.cdf(0.5) - u.cdf(0.5))

**`0.4` and `0.0`.**

The first is a range, so it has an area and therefore a probability.

The second is the *same point twice* — a range of zero width. Zero width means zero area means
zero probability, even though 0.5 is a perfectly possible value.

This is the thing to watch for in your own code: if a continuous probability comes back as
exactly `0.0`, check whether you accidentally asked about a point.

In [ ]:
print("shrinking the window around 0.5:")
for w in [0.1, 0.01, 0.001, 0.0001, 0]:
    print(f"  width {w:<8} P = {u.cdf(0.5+w/2) - u.cdf(0.5-w/2):.6f}")
print()
print("it only reaches 0 when the width does")

### A8

```python
print(np.random.default_rng(99).random(2))
print(np.random.default_rng(99).random(2))
```

Same or different? And is that what you want?

In [ ]:
print(np.random.default_rng(99).random(2))
print(np.random.default_rng(99).random(2))

**Identical.** Same seed, same sequence, every single time.

These numbers are *pseudorandom*: a deterministic formula producing a sequence that passes the
uniformity tests. Statistically uniform, not unpredictable.

That is usually exactly what you want — your results reproduce, a failing test fails the same
way twice, and a reviewer can rerun your notebook and get your numbers.

**But never use it for anything security-related.** Given enough output the sequence can be
worked backwards and every future value predicted. Use the `secrets` module for tokens, keys
and passwords.

In [ ]:
import secrets
print("seeded (reproducible):")
print(" ", np.random.default_rng(99).random(2))
print(" ", np.random.default_rng(99).random(2))
print()
print("secrets (not reproducible, no seed):")
print(" ", secrets.token_hex(8))
print(" ", secrets.token_hex(8))

---
# Part B — Spot & fix the bug

Each cell runs without error and gives a **wrong** answer. Find the bug, say what the wrong
number means, and write the fix.

### B1 — the width trap

A delivery promise says a parcel arrives between **14:00 and 18:00**, equally likely at any
moment. What is the chance it arrives before 15:00?

In [ ]:
arrival = stats.uniform(14, 18)
print(f"P(before 15:00) = {arrival.cdf(15):.4f}")

**Bug:** `stats.uniform` takes `loc` and **scale (width)**, not two endpoints.
`uniform(14, 18)` is U(14, **32**) — a 32-hour delivery window.

**The wrong number** (0.0312) is the chance of arriving in the first hour of a *32-hour*
window, not a 4-hour one.

**Fix:** `stats.uniform(14, 4)` — start at 14, width 4. The right answer is **0.25**, which you
could have got in your head: one hour out of four.

That head-check is the real lesson. 0.0312 should have looked wrong immediately.

In [ ]:
wrong = stats.uniform(14, 18)
right = stats.uniform(14, 4)

print(f"wrong: spans {wrong.ppf([0,1])}  P(before 15:00) = {wrong.cdf(15):.4f}")
print(f"right: spans {right.ppf([0,1])}  P(before 15:00) = {right.cdf(15):.4f}")
print()
print("sanity check you can do in your head: 1 hour out of 4 = 0.25")
print(f"mean arrival time: wrong {wrong.mean()}:00, right {right.mean()}:00")

### B2 — the point probability

A machine cuts rods to a length uniform between 99.5 and 100.5 mm. The spec says a rod is
acceptable if it is **exactly 100 mm**. What fraction pass?

In [ ]:
rod = stats.uniform(99.5, 1.0)
passes = rod.cdf(100) - rod.cdf(100)
print(f"Fraction passing = {passes:.6f}")
print("-> nothing ever passes. The machine is useless.")

**Bug:** not in the code — in the **question**. "Exactly 100 mm" is a point on a continuous
variable, so its probability is 0 by definition. The code is right and the conclusion is
nonsense.

**The wrong number means** nothing is wrong with the machine; the specification is impossible
as written. No machine can cut to infinite precision, and no instrument can measure it.

**Fix:** a real spec is a **tolerance**. "100 mm ± 0.1" has a sensible answer: 20% of rods pass.

This is the practical face of P(X = x) = 0 — every real-world specification on a continuous
quantity must be a range, and if yours is not, it is unmeasurable.

In [ ]:
rod = stats.uniform(99.5, 1.0)
print("the spec as written:", rod.cdf(100) - rod.cdf(100))
print()
print("with a realistic tolerance:")
for tol in [0.5, 0.25, 0.1, 0.05, 0.01]:
    p = rod.cdf(100 + tol) - rod.cdf(100 - tol)
    print(f"  100 +/- {tol:<5} ->  {p:.1%} pass")
print()
print("Every spec on a continuous quantity has to be a range.")

### B3 — proving the wrong thing

A team checks whether their randomiser is producing uniform values on [0, 1].

In [ ]:
rng_b3 = np.random.default_rng(21)
values = rng_b3.beta(2, 2, 3000)        # what the randomiser actually produced

print(f"min  {values.min():.4f}   max  {values.max():.4f}")
print(f"mean {values.mean():.4f}  (target 0.5)")
print("-> spans essentially the full range and centres on 0.5. Looks uniform.")

**Bug:** the bounds and the mean say nothing about the **shape**. A Beta(2,2) also runs 0 to 1
(this sample reaches 0.006 and 0.996) and also averages 0.5 — it is just clustered in the
middle.

**The wrong conclusion** is that the randomiser is fine. It is producing middling values far
too often and extreme values almost never.

**Fix:** look at the spread, or run a test, or — fastest of all — draw a histogram.

- sd of a true U(0,1) is **0.2887**. This data has **0.225**, about 78% of it.
- `stats.kstest(values, 'uniform')` returns p ≈ 1e-26, while the genuine sample gives 0.69.

Note how weak the sd signal is: 0.225 against 0.289 is not the kind of gap that leaps out of a
summary table. The histogram is what makes it obvious in one glance.

In [ ]:
rng_b3 = np.random.default_rng(21)
values = rng_b3.beta(2, 2, 3000)
truly  = np.random.default_rng(21).random(3000)

print(f"{'':<10} {'min':>6} {'max':>6} {'mean':>8} {'sd':>8} {'KS p':>11}")
for name, d in [("suspect", values), ("real U(0,1)", truly)]:
    print(f"{name:<10} {d.min():>6.3f} {d.max():>6.3f} {d.mean():>8.4f} "
          f"{d.std():>8.4f} {stats.kstest(d, 'uniform').pvalue:>11.2e}")
print(f"\ntheory sd for U(0,1) = {1/np.sqrt(12):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 2.8), sharey=True)
for ax, d, t in [(axes[0], values, "the suspect randomiser"), (axes[1], truly, "really uniform")]:
    ax.hist(d, bins=25, color="#3E7CB1")
    ax.axhline(len(d)/25, color="#E8674C", ls="--", lw=2)
    ax.set_title(t)
plt.tight_layout(); plt.show()

### B4 — "we tested it"

A casino checks one of its dice for fairness and reports back.

In [ ]:
rng_b4 = np.random.default_rng(2)
true_p = np.array([1, 1, 1, 1, 1, 2]) / 7     # this die really IS crooked

rolls  = rng_b4.choice(np.arange(1, 7), 60, p=true_p)
counts = np.bincount(rolls, minlength=7)[1:]
p = stats.chisquare(counts).pvalue

print("counts:", counts)
print(f"chi-square p = {p:.3f}")
print("-> p > 0.05, no evidence of bias. The die is fair.")

**Bug:** the conclusion. "No evidence of bias" is not "the die is fair" — and with only 60
rolls the test had almost no chance of finding the bias even though it is blatant.

This die lands on 6 **twice as often as it should** (28.6% vs 16.7%). Simulate the whole
procedure and a 60-roll test catches that only about **40%** of the time.

**The wrong conclusion** could lose a casino a great deal of money, or clear a cheat.

**Fix:** before trusting a negative result, ask *how much data would I have needed?* Here you
need around **150 rolls for an 80% chance** of catching it, and 300 for near-certainty.

Same lesson as Day 25's A/B test: a null result is only meaningful alongside its sample size.

In [ ]:
rng_b4 = np.random.default_rng(99)
true_p = np.array([1, 1, 1, 1, 1, 2]) / 7

print(f"{'rolls':>7} {'catches it':>12}")
ns, power = [], []
for n in [40, 60, 100, 150, 200, 300, 500]:
    hits = sum(stats.chisquare(
                   np.bincount(rng_b4.choice(np.arange(1, 7), n, p=true_p), minlength=7)[1:]
               ).pvalue < 0.05
               for _ in range(500))
    ns.append(n); power.append(100 * hits / 500)
    print(f"{n:>7} {100*hits/500:>11.0f}%")

print("\nWith 60 rolls you would miss this cheat about 6 times in 10.")
print("Around 150 rolls gets you to the usual 80% bar.")

fig, ax = plt.subplots(figsize=(7.5, 2.8))
ax.axhspan(0, 80, color="#FDECE8")
ax.plot(ns, power, color="#3E7CB1", lw=2.5, marker="o")
ax.axhline(80, color="#152238", ls="--", lw=1.8)
ax.set_xlabel("rolls"); ax.set_ylabel("catches it (%)"); ax.set_ylim(0, 105)
plt.show()

---
# Part C — Write the code

Write the code yourself. Then say the answer in plain English.

### C1 — the lift

A lift arrives every **4 minutes**. You walk up without checking, so your wait is uniform.

1. What is your average wait, and the standard deviation?
2. What fraction of the time do you wait **less than 1 minute**?
3. What fraction **more than 3 minutes**?
4. Your colleague says "the lift takes 2 minutes on average, so it is fine". What does that
   miss?

In [ ]:
lift = stats.uniform(0, 4)        # loc=0, width=4

print(f"1.  mean wait = {lift.mean()} min,  sd = {lift.std():.4f} min")
print(f"2.  P(wait < 1) = {lift.cdf(1):.4f}")
print(f"3.  P(wait > 3) = {1 - lift.cdf(3):.4f}")
print()
print("4.  A quarter of the time you wait over 3 minutes - which is")
print("    what people actually complain about. The average hides both tails.")

- mean 2 min, sd 1.155 min (= 4/√12)
- P(wait < 1) = **0.25**
- P(wait > 3) = **0.25**

What the average misses: **one trip in four** means waiting over 3 minutes. That is the
experience people complain about, and "2 minutes on average" describes none of it.

Same point as the bus on the slides, and the same point as Day 25's call centre. An average
describes the typical case; the distribution describes the cases that annoy people.

### C2 — rounding error

Generate 100,000 prices uniformly between 0 and 1000, round each to the nearest **£1**, and
examine the rounding errors.

1. What range do the errors cover?
2. What is their mean and standard deviation? Compare with theory.
3. Plot a histogram. Is it uniform?
4. If you round 10,000 prices and sum them, roughly how far off is the total?

In [ ]:
rng_c2 = np.random.default_rng(262)
prices = rng_c2.random(100_000) * 1000
err = prices - np.round(prices)

print(f"1.  range {err.min():.4f} to {err.max():.4f}   (exactly -0.5 to 0.5)")
print(f"2.  mean {err.mean():.5f}  (theory 0)")
print(f"    sd   {err.std():.5f}  (theory 1/sqrt(12) = {1/np.sqrt(12):.5f})")

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.hist(err, bins=50, color="#2FA98C")
ax.set_title("3.  Rounding error is uniform - provably, not approximately")
plt.show()

#  4. errors are independent, so variances add
n = 10_000
total_sd = np.sqrt(n) * (1/np.sqrt(12))
print(f"4.  summing {n} rounded prices: the total is off by about "
      f"+/- {total_sd:.1f}")
print(f"    (sqrt({n}) x {1/np.sqrt(12):.4f}  - errors cancel, but only as sqrt(n))")

1. Exactly **−0.5 to 0.5**, by construction.
2. mean ≈ 0, sd ≈ **0.2887** = 1/√12. The theory is exact, not approximate.
3. Flat. This is one of the few places where uniformity is a matter of arithmetic rather than
   an assumption.
4. The total is off by about **±29**. The individual errors cancel — but only as √n, not as n.
   Round 10,000 prices and the accumulated error is roughly 29, not 0 and not 5,000.

Point 4 is why financial systems do not round intermediate values. The errors are unbiased,
which sounds safe, and they still accumulate.

### C3 — is this dataset uniform?

Three datasets are generated below. For each one:

1. State whether you think it is uniform, from a histogram alone.
2. Back it up with the right test (`chisquare` for discrete, `kstest` for continuous).
3. For any that fail, say what shape it actually has.

In [ ]:
rng_c3 = np.random.default_rng(263)
a = rng_c3.random(3000)                                   # continuous
b = rng_c3.triangular(0, 0.5, 1, 3000)                    # continuous
c = rng_c3.choice(np.arange(1, 11), 3000,
                  p=np.r_[np.full(9, 0.09), 0.19])        # discrete 1-10

fig, axes = plt.subplots(1, 3, figsize=(13, 2.8))
for ax, d, t, bins in [(axes[0], a, "a", 25), (axes[1], b, "b", 25),
                       (axes[2], c, "c", np.arange(0.5, 11.5))]:
    ax.hist(d, bins=bins, color="#3E7CB1")
    ax.set_title(t)
plt.tight_layout(); plt.show()

print("a  continuous, KS p =", f"{stats.kstest(a, 'uniform').pvalue:.3f}")
print("b  continuous, KS p =", f"{stats.kstest(b, 'uniform').pvalue:.2e}")
obs = np.bincount(c, minlength=11)[1:]
print("c  discrete, counts", obs)
print("   chi-square p =", f"{stats.chisquare(obs).pvalue:.2e}")
print()
print(f"sd check:  a {a.std():.4f}   b {b.std():.4f}   theory for U(0,1) {1/np.sqrt(12):.4f}")

- **a** is uniform. KS p ≈ 0.4, sd ≈ 0.29, the histogram is flat.
- **b** is **not**. It is a *triangular* distribution — same range 0 to 1, same mean 0.5, but
  values pile up in the middle. KS p is effectively zero and the sd is about 0.20 against a
  theoretical 0.2887.
- **c** is **not**. Nine values at 9% and one at 19% — face 10 appears about twice as often as
  the rest. A chi-square test catches it easily at n = 3000.

The thing to notice: **b** passes every check you can do without looking at the shape. Same
range, same mean. The histogram and the sd are what give it away.

### C4 — build your own sampler

Using **only** `rng.random()` — no `rng.choice`, no `rng.normal`, no `rng.poisson`:

1. Sample 100,000 values from U(20, 50).
2. Sample 100,000 values from an exponential with mean 5.
   *Hint: its inverse cdf is* `-mean * log(1 - u)`.
3. Sample 100,000 rolls of a die that lands on 1 half the time and on 2–6 equally otherwise.
4. Check each against the distribution you were aiming for.

In [ ]:
rng_c4 = np.random.default_rng(264)

#  --- 1. U(20, 50): stretch and shift
u = rng_c4.random(100_000)
scaled = 20 + u * (50 - 20)
print(f"1.  U(20,50): min {scaled.min():.2f} max {scaled.max():.2f} "
      f"mean {scaled.mean():.3f} (target 35)")

#  --- 2. exponential with mean 5
u = rng_c4.random(100_000)
expo = -5 * np.log(1 - u)
print(f"2.  Exp(mean 5): mean {expo.mean():.3f}  sd {expo.std():.3f} "
      f"(both target 5)")

#  --- 3. a die that favours 1
probs = np.array([.5, .1, .1, .1, .1, .1])
cum = np.cumsum(probs)
u = rng_c4.random(100_000)
rolls = np.searchsorted(cum, u) + 1
freq = np.bincount(rolls, minlength=7)[1:] / len(rolls)
print(f"3.  loaded die: asked {probs}")
print(f"                 got  {freq.round(4)}")

#  --- 4. check 2 against scipy
print(f"\n4.  scipy Exp(scale=5): mean {stats.expon(scale=5).mean()} "
      f"sd {stats.expon(scale=5).std()}")

All three land on target.

- U(20, 50): a uniform is just `a + u × (b − a)`. No inverse cdf needed, because the uniform
  *is* its own starting point.
- Exponential: mean and sd both come out at 5, which is correct — the exponential has mean =
  sd, the same way the Poisson has mean = variance.
- The loaded die: `searchsorted` on the cumulative probabilities is the discrete version of an
  inverse cdf, and the band widths *are* the probabilities.

This is the whole of section 4 in one cell: a single stream of uniform numbers, reshaped three
different ways.

### C5 — a real decision

A support team's tickets arrive during an 8-hour day. Historically the arrival time within the
day is roughly uniform.

1. If 40 tickets arrive per day, how many land in the last hour, on average?
2. The team wants cover for the busiest hour. Under a uniform model, is there a busiest hour?
3. Simulate 1,000 days and find the **maximum** tickets in any single hour, each day. What is
   the 95th percentile of that maximum?
4. So how many should they staff for, and why is the answer not 5?

In [ ]:
rng_c5 = np.random.default_rng(265)
per_day, hours = 40, 8

print(f"1.  {per_day}/{hours} = {per_day/hours:.1f} tickets per hour on average")
print("2.  Under a uniform model, no - every hour has the same EXPECTED load.")
print("    But 'same expected load' is not 'same load'.")

#  --- 3. simulate
days = 1000
#  each ticket lands in a uniformly chosen hour
hour_of = rng_c5.integers(0, hours, size=(days, per_day))
counts = np.stack([np.bincount(d, minlength=hours) for d in hour_of])
busiest = counts.max(axis=1)

print(f"\n3.  busiest hour of the day, over {days} days:")
print(f"      mean {busiest.mean():.2f}")
print(f"      95th percentile {np.percentile(busiest, 95):.0f}")
print(f"      worst seen {busiest.max()}")

print(f"\n4.  Staff for {np.percentile(busiest, 95):.0f}, not 5.")
print("    5 is the average hour. The busiest hour of a day is a maximum,")
print("    and a maximum of 8 draws runs well above the average of one.")

1. **5 per hour** on average.
2. Under a uniform model no hour is *systematically* busier — but that is not the question the
   team is asking. Equal expected load does not mean equal load.
3. Simulating 1,000 days, the busiest hour averages about **8** tickets and its 95th percentile
   is **11**.
4. **Staff for about 11, not 5.**

The reasoning is the interesting part. 5 is the average of *one* hour. The busiest hour is the
**maximum of 8 hours**, and a maximum sits well above an average — that is what a maximum is.
Staffing to the mean guarantees the worst hour is under-covered every single day.

This is the uniform-distribution version of Day 25's staffing build, and it is the same
mistake in a different costume: using a central value to answer a question about an extreme.

---
# Part D — Challenge

### D1 — watch a flat line become a bell curve

This is a preview of Day 27, and you can discover the whole result yourself.

1. Draw 200,000 samples each of: one U(0,1), the sum of two, the sum of three, and the sum of
   twelve.
2. For each, report the mean, sd, **skew** and **kurtosis**. (A normal distribution has skew 0
   and kurtosis 0.)
3. Plot all four with a normal curve overlaid, using mean = n/2 and sd = √(n/12).
4. The sum of twelve U(0,1) has a notable property. What is its standard deviation, and why is
   that convenient?
5. In one sentence: what do you think Day 27 is going to say?

In [ ]:
rng_d1 = np.random.default_rng(2606)

print(f"{'n':>3} {'mean':>8} {'sd':>8} {'skew':>9} {'kurtosis':>10}")
for n in [1, 2, 3, 12]:
    s = rng_d1.random((200_000, n)).sum(axis=1)
    print(f"{n:>3} {s.mean():>8.4f} {s.std():>8.4f} "
          f"{stats.skew(s):>+9.4f} {stats.kurtosis(s):>+10.4f}")
print()
print("A normal distribution has skew 0 and kurtosis 0.")
print("Watch the kurtosis climb from -1.20 towards 0.")

In [ ]:
rng_d1 = np.random.default_rng(2606)
fig, axes = plt.subplots(1, 4, figsize=(13, 2.9))
for ax, n in zip(axes, [1, 2, 3, 12]):
    s = rng_d1.random((120_000, n)).sum(axis=1)
    ax.hist(s, bins=70, color="#3E7CB1", density=True)
    g = np.linspace(s.min(), s.max(), 300)
    ax.plot(g, stats.norm.pdf(g, n/2, np.sqrt(n/12)), color="#152238", lw=2)
    ax.set_title(f"sum of {n}", fontsize=10); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
#  4. why twelve in particular?
n = 12
print(f"variance of one U(0,1) = 1/12 = {1/12:.6f}")
print(f"variance of the sum of {n}  = {n}/12 = {n/12:.1f}")
print(f"so sd = {np.sqrt(n/12):.1f}  -  exactly 1")
print()
print("mean = 12/2 = 6, sd = 1")
print("so  sum(12 uniforms) - 6  is almost exactly a standard normal.")
print()
rng_d1 = np.random.default_rng(1)
approx = rng_d1.random((200_000, 12)).sum(axis=1) - 6
real   = np.random.default_rng(1).normal(0, 1, 200_000)
print(f"{'':<12} {'mean':>9} {'sd':>8} {'skew':>9} {'kurtosis':>10} {'P(>3)':>9}")
for name, d in [("12-uniform", approx), ("real normal", real)]:
    print(f"{name:<12} {d.mean():>+9.4f} {d.std():>8.4f} {stats.skew(d):>+9.4f} "
          f"{stats.kurtosis(d):>+10.4f} {(d > 3).mean():>9.5f}")
print(f"{'theory':<12} {0:>+9.4f} {1:>8.4f} {0:>+9.4f} {0:>+10.4f} "
      f"{1 - stats.norm.cdf(3):>9.5f}")

**1 & 2.** The numbers:

| n | mean | sd | skew | kurtosis |
|---|---|---|---|---|
| 1 | 0.500 | 0.289 | 0.00 | **−1.20** |
| 2 | 1.000 | 0.408 | 0.00 | −0.60 |
| 3 | 1.499 | 0.500 | 0.00 | −0.41 |
| 12 | 6.002 | 1.000 | 0.00 | **−0.11** |

Skew is zero throughout, because the uniform is symmetric and stays symmetric. The interesting
column is **kurtosis**: it starts at −1.20 (the exact value for a uniform — flat, no tails) and
climbs steadily towards 0, which is the normal's value.

**3.** The overlaid curve is already close at n = 3 and indistinguishable at n = 12.

**4.** The variance of one U(0,1) is 1/12, so the variance of a sum of twelve is 12/12 = **1**,
and the sd is **exactly 1**. The mean is 6. So `sum(12 uniforms) − 6` is an almost perfect
standard normal.

That is not a coincidence of this exercise — it is an old trick. Before fast normal generators
existed, this is how machines produced normal numbers. Its one real flaw is the tails: it can
never produce a value beyond ±6, so for anything that cares about extreme events it is wrong
in exactly the place that matters.

**5.** The answer Day 27 gives: **when you add up a lot of independent random things, the
total is normal — whatever the things were.** It does not matter that the uniform is flat. It
does not matter that the binomial is lumpy or the Poisson skewed. Add enough of them and you
get a bell curve, every time.

That is the Central Limit Theorem, and you have now watched it happen three times without
anyone naming it.

### D2 — audit a dataset

Below is a set of expense claims. Somebody suspects a portion of them were invented.

1. Check whether the **leading digits** follow Benford's law or are suspiciously uniform.
2. Do the same for the **final two digits** (the pence). What would you expect for honest data,
   and what do you see?
3. State your conclusion and — this matters — **say what would change your mind**.

There is no answer printed anywhere. You have to decide.

In [ ]:
rng_d2 = np.random.default_rng(2626)

#  1,200 expense claims. Some genuine, some not. You are not told which.
real = np.round(10 ** (rng_d2.random(900) * 3.5), 2)
fake = rng_d2.integers(20, 950, 300).astype(float)

claims = np.concatenate([real, fake])
rng_d2.shuffle(claims)

print("first 12 claims:", claims[:12])
print("n =", len(claims), " total =", round(claims.sum(), 2))

In [ ]:
#  --- 1. leading digits
lead = np.array([int(f"{v:.10e}"[0]) for v in claims])
obs = np.bincount(lead, minlength=10)[1:]
benford = np.log10(1 + 1/np.arange(1, 10))
expected_b = benford * len(claims)
expected_u = np.full(9, len(claims)/9)

print(f"{'digit':>6} {'observed':>9} {'Benford':>9} {'uniform':>9}")
for d in range(1, 10):
    print(f"{d:>6} {obs[d-1]:>9} {expected_b[d-1]:>9.1f} {expected_u[d-1]:>9.1f}")

print()
print(f"chi-square vs Benford : p = {stats.chisquare(obs, expected_b).pvalue:.2e}")
print(f"chi-square vs uniform : p = {stats.chisquare(obs, expected_u).pvalue:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
d = np.arange(1, 10)
ax.bar(d - 0.27, obs/len(claims), width=0.27, color="#3E7CB1", label="observed")
ax.bar(d,        benford,         width=0.27, color="#2FA98C", label="Benford")
ax.bar(d + 0.27, np.full(9, 1/9), width=0.27, color="#9AA7B8", label="uniform")
ax.set_xticks(d); ax.set_xlabel("leading digit"); ax.set_ylabel("proportion")
ax.legend(frameon=False); ax.set_title("Leading digits of the expense claims")
plt.show()

In [ ]:
#  --- 2. the pence
pence = np.round((claims * 100) % 100).astype(int)
obs_p = np.bincount(pence, minlength=100)

print(f"distinct pence values present: {(obs_p > 0).sum()} out of 100")
print(f"share ending in .00 : {(pence == 0).mean():.3f}   (uniform would be 0.01)")
print(f"share ending in .50 : {(pence == 50).mean():.3f}")
print(f"chi-square vs uniform over 00-99 : p = {stats.chisquare(obs_p).pvalue:.3f}")

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.bar(np.arange(100), obs_p, color="#3E7CB1", width=1.0)
ax.axhline(len(claims)/100, color="#E8674C", ls="--", lw=2)
ax.set_xlabel("pence"); ax.set_ylabel("count")
ax.set_title("Final two digits - uniform is what honest data looks like here")
plt.show()

**1. The leading digits: nothing.** The observed counts fit Benford comfortably (chi-square
p ≈ 0.56) and are nowhere near uniform (p ≈ 1e-116). The famous test clears the dataset.

This is worth sitting with, because the exercise was set up to look like a Benford problem. The
reason it finds nothing is that the invented claims were drawn from £20 to £950 — a range
spanning more than one order of magnitude — and **a uniform draw across orders of magnitude
produces roughly Benford-ish leading digits by itself**. Benford catches people who invent
digits; it does not catch people who invent plausible amounts.

**2. The pence: everything.** Honest prices have no reason to favour any particular pence
value, so uniform *is* the right expectation here — and the data violates it enormously.
**26% of claims end in `.00`**, against the 1% you would expect. The chi-square p-value is
effectively zero.

Note that this inverts the slide's framing, and deliberately so: **uniform is suspicious for
leading digits and expected for trailing ones.** The same test points opposite ways depending
on which digits you run it on, and here it is the boring check nobody thinks of that finds the
problem.

**3. The conclusion, and what would change it.**

> *The leading digits are consistent with genuine data. But a quarter of these claims are
> whole pounds, against an expected one in a hundred. That is the finding, and it is grounds
> to pull those records and look at them — not grounds to accuse anyone.*

What would change my mind, in either direction:

- **Towards innocent:** if the expense policy contains flat amounts — a £50 per-diem, a £25
  mileage allowance, a fixed room rate — then round numbers are exactly what honest claims look
  like. **Check the policy before the data.** This single explanation could account for all of
  it.
- **Towards innocent:** if the round claims are spread evenly across every claimant and month,
  it is a feature of the process rather than of a person.
- **Towards guilty:** if they concentrate in one claimant, one period, or cluster just under an
  approval threshold.
- **Towards guilty:** if the round claims also lack receipts, or repeat the same amount.

**And the methodological point**, which is most of the marks: I ran two tests and reported
both, including the one that found nothing. Reporting only the pence result would make this
look like a dataset screaming fraud, when the better-known test on the same data says it is
fine. A test that produces a lead is not a verdict, and this one is pointed at named
individuals, where being casually wrong is expensive.

---
# Where the marks were

| Question | The thing being tested |
|---|---|
| A1, A2 | pdf is a density, not a probability — and can exceed 1 |
| A3 | ⚠️ `uniform(loc, WIDTH)`, not two endpoints |
| A4, A5 | ⚠️ three libraries, three conventions on the upper bound |
| A6 | sd of U(0,1) is 1/√12 = 0.2887 |
| A7 | a point has zero width, so zero probability |
| A8 | pseudorandom: reproducible is a feature, except for secrets |
| B1 | the width trap, caught by a head-check |
| B2 | a continuous spec must be a tolerance, not a point |
| B3 | same range and mean, wrong shape |
| B4 | a null result is meaningless without its sample size |
| C1–C3 | modelling, then actually checking the assumption |
| C4 | building any distribution from `rng.random()` alone |
| C5 | a maximum is not an average |
| D1 | the Central Limit Theorem, discovered rather than told |
| D2 | using a test to produce a lead, not a verdict |

If you only remember three things from today:

1. **Probability is area.** That is why P(X = exactly x) = 0 and why a density can exceed 1.
2. **`U(a, b)` is `stats.uniform(a, b − a)`.** The second argument is a width.
3. **Uniform is not the same as random.** Benford is the proof, and the leading digits of real
   data are nowhere near flat.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*